# Land Registry Price Data (National 2025) Structural Audit

#### Load data

In [1]:
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

column_names = [
    "transaction_id",
    "price",
    "date_of_transfer",
    "postcode",
    "property_type",
    "old_new",
    "duration",
    "paon",
    "saon",
    "street",
    "locality",
    "town_city",
    "district",
    "county",
    "ppd_category_type",
    "record_status"
]
df = pd.read_csv("../data/raw/pp-2025.csv", names=column_names)

#### Investigating anomalies

Checking datatypes

In [2]:
df.dtypes

transaction_id         str
price                int64
date_of_transfer       str
postcode               str
property_type          str
old_new                str
duration               str
paon                   str
saon                   str
street                 str
locality               str
town_city              str
district               str
county                 str
ppd_category_type      str
record_status          str
dtype: object

Looking at data consistency with types

In [3]:
df.head(5)

,transaction_id,price,date_of_transfer,postcode,property_type,old_new,duration,paon,saon,street,locality,town_city,district,county,ppd_category_type,record_status
0,{42C129E4-C259-60A9-E063-4804A8C0C25D},635000,2025-05-14 00:00,LN11 8GN,D,N,F,2,NaN,KENWICK VIEW,NaN,LOUTH,EAST LINDSEY,LINCOLNSHIRE,A,A
1,{42C129E4-C25B-60A9-E063-4804A8C0C25D},230000,2025-08-29 00:00,PE10 2BL,S,N,F,6,NaN,CONWAY DRIVE,NaN,BOURNE,SOUTH KESTEVEN,LINCOLNSHIRE,A,A
2,{42C129E4-C25E-60A9-E063-4804A8C0C25D},185000,2025-03-27 00:00,LN10 6RJ,S,N,F,6,NaN,KING EDWARD AVENUE,NaN,WOODHALL SPA,EAST LINDSEY,LINCOLNSHIRE,A,A
3,{42C129E4-C260-60A9-E063-4804A8C0C25D},76000,2025-06-06 00:00,NG31 8SP,T,N,L,5,NaN,RAWLE GREEN,NaN,GRANTHAM,SOUTH KESTEVEN,LINCOLNSHIRE,A,A
4,{42C129E4-C261-60A9-E063-4804A8C0C25D},237500,2025-09-18 00:00,PE11 4PP,S,N,F,4,NaN,PLATINUM CLOSE,GOSBERTON,SPALDING,SOUTH HOLLAND,LINCOLNSHIRE,A,A


Checking null and unique values

In [4]:

print(df.nunique())
print(df.isna().sum())
print(df.duplicated().sum())

transaction_id       954145
price                 30201
date_of_transfer        364
postcode             548539
property_type             5
old_new                   2
duration                  2
paon                  80138
saon                   7624
street               185840
locality              16521
town_city              1146
district                318
county                  113
ppd_category_type         2
record_status             1
dtype: int64
transaction_id            0
price                     0
date_of_transfer          0
postcode               2339
property_type             0
old_new                   0
duration                  0
paon                      0
saon                 840679
street                15688
locality             587665
town_city                 0
district                  0
county                    0
ppd_category_type         0
record_status             0
dtype: int64
0


Looking at min/max price

In [5]:
print(df['price'].sort_values())
print((df['price'] < 1000).sum())
print((df['price'] > 100000000).sum())

603783            1
322683            1
322684            1
322682            1
322685            1
            ...    
34047     127700000
117050    172100000
76356     190498163
77515     260000000
133436    793020000
Name: price, Length: 954145, dtype: int64
540
8


Inspect values of street, town_city, locality, district, county

Searching for unexpected characters

In [6]:
to_check = ['street', 'town_city', 'locality', 'district', 'county']
for n in to_check:
    print(f"Checking {n} : contains {df[n].str.count(r"[^&a-zA-Z0-9'.\s-]").sum()} special or accented characters excl -'&.")


Checking street : contains 11.0 special or accented characters excl -'&.
Checking town_city : contains 0 special or accented characters excl -'&.
Checking locality : contains 9.0 special or accented characters excl -'&.
Checking district : contains 6668 special or accented characters excl -'&.
Checking county : contains 6668 special or accented characters excl -'&.


Testing regex search

In [7]:
import re
test = "!£$%^&*()<>?/@|¬"

re.findall(r"[^&a-zA-Z0-9'.\s-]", test)

['!', '£', '$', '%', '^', '*', '(', ')', '<', '>', '?', '/', '@', '|', '¬']

Looking at actual flagged values

In [8]:
for n in to_check:
    print(df[n][df[n].str.contains(r"[^&a-zA-Z0-9'.\s-]")].value_counts())


street
FOURTH STREET, WATLING STREET BUNGALOWS    5
OAKLEIGH SQUARE, HAMMOND WAY               2
CHESTNUT GRANGE, BLACKMAN COURT            2
EAST MOOR PARK, CUFFERN                    1
MILLERS VIEW, WINDMILL WAY                 1
Name: count, dtype: int64
Series([], Name: count, dtype: int64)
locality
WESTWARD HO!    9
Name: count, dtype: int64
district
BOURNEMOUTH, CHRISTCHURCH AND POOLE    6668
Name: count, dtype: int64
county
BOURNEMOUTH, CHRISTCHURCH AND POOLE    6668
Name: count, dtype: int64


A list of the unique special characters being found

In [9]:

results = set()
for n in to_check:
    flagged_rows = df[n][df[n].str.contains(r"[^&a-zA-Z0-9'.\s-]")]
    for r in flagged_rows:
       matches = re.findall(r"[^&a-zA-Z0-9'.\s-]", r)
       results.update(matches)
print(results)



{',', '!'}


Finding which day of the year had no sales

In [10]:
# print(df['date_of_transfer'].sort_values())
df['date_of_transfer'] = df['date_of_transfer'].astype('datetime64[s]')
dates = pd.DataFrame()
dates['day'] = df['date_of_transfer'].dt.day
dates['month'] = df['date_of_transfer'].dt.month
print(dates.groupby('month')['day'].nunique())
print(dates[dates['month'] == 12].groupby('day').count())

month
1     31
2     28
3     31
4     30
5     31
6     30
7     31
8     31
9     30
10    31
11    30
12    30
Name: day, dtype: int64
     month
day       
1     3108
2     1855
3     2157
4     2504
5     7709
6        5
7        8
8     2647
9     2068
10    2948
11    3098
12    9316
13       5
14       8
15    4094
16    3094
17    4120
18    5841
19   11376
20       2
21       4
22    1387
23     389
24      38
25       3
26       2
28       1
29      25
30      23
31      35


No sales on 27th December

Postcode check on structure

In [11]:
print(f"Postcodes between {df['postcode'].str.len().max()} and {df['postcode'].str.len().min()} characters in length")


passed_postcodes = df['postcode'].str.match(r"^[A-Z]{1,2}\d[A-Z0-9]? \d[A-Z]{2}$", na=False).sum()
total_postcodes = df['postcode'].dropna().count()
print(f"{total_postcodes - passed_postcodes} postcodes failed the regex check")
print(df['postcode'][~df['postcode'].str.match(r"^[A-Z]{1,2}\d[A-Z0-9]? \d[A-Z]{2}$", na=True)])

Postcodes between 8.0 and 6.0 characters in length
0 postcodes failed the regex check
Series([], Name: postcode, dtype: str)
